In [6]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] ='0'

In [7]:
!pip install transformers trl peft accelerate Dataset bitsandbytes

In [8]:
from peft import (
    LoraConfig,
    PeftModel,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    pipeline,
)
from datasets import Dataset, load_dataset
from trl import DataCollatorForCompletionOnlyLM, SFTConfig, SFTTrainer
import torch

In [14]:
PAD_TOKEN = "<|pad|>"
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
tokenizer.add_special_tokens({"pad_token": PAD_TOKEN})
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    #device_map="auto",
)
device = torch.device("cuda")
model.to(device)
model.resize_token_embeddings(len(tokenizer), pad_to_multiple_of=8)

`low_cpu_mem_usage` was None, now default to True since model is quantized.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Embedding(32016, 3072, padding_idx=32000)

In [19]:
input = "what is machine learninig"
input_token = tokenizer.encode(input, return_tensors="pt", truncation=True, padding=True, max_length=128)
input_token = input_token.to('cuda')
output_token = model.generate(input_token, max_length=128)
output_text = tokenizer.decode(output_token[0], skip_special_tokens=True)
print(output_text)

what is machine learninig?

# Answer
Machine learning is a subset of artificial intelligence (AI) that focuses on building systems that can learn from and make decisions based on data. It involves the development of algorithms that can learn from and make predictions or decisions based on input data, without being explicitly programmed to perform the task. Machine learning algorithms use statistical methods to enable computers to improve at a task with experience. This field has applications across various industries, including finance, healthcare, transportation, and more, where it can be used for tasks such as fraud detection, disease diagnosis,


In [20]:
lora_config = LoraConfig(
    r=32,
    lora_alpha = 32,
    lora_dropout = 0.05,
    target_modules=[
        "self_attn.q_proj",
        "self_attn.k_proj",
        "self_attn.v_proj",
        "self_attn.o_proj",
        "mlp.gate_proj",
        "mlp.up_proj",
        "mlp.down_proj",
    ],
    bias="none",
    task_type="CAUSAL_LM"
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)
device = torch.device("cuda")
model.cuda()
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Phi3ForCausalLM(
      (model): Phi3Model(
        (embed_tokens): Embedding(32016, 3072, padding_idx=32000)
        (layers): ModuleList(
          (0-31): 32 x Phi3DecoderLayer(
            (self_attn): Phi3Attention(
              (o_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
             

In [21]:
model.print_trainable_parameters()

trainable params: 17,825,792 || all params: 3,838,610,432 || trainable%: 0.4644


In [22]:
df = load_dataset("MakTek/Customer_support_faqs_dataset",split="train")
df

Dataset({
    features: ['question', 'answer'],
    num_rows: 200
})

In [23]:
df.to_pandas()

,question,answer
0,How can I create an account?,"To create an account, click on the 'Sign Up' b..."
1,What payment methods do you accept?,"We accept major credit cards, debit cards, and..."
2,How can I track my order?,You can track your order by logging into your ...
3,What is your return policy?,Our return policy allows you to return product...
4,Can I cancel my order?,You can cancel your order if it has not been s...
...,...,...
195,Do you offer a satisfaction guarantee?,"Yes, we offer a satisfaction guarantee on our ..."
196,How can I apply for a job at your company?,"To apply for a job at our company, visit our C..."
197,What is the warranty on your products?,The warranty on our products varies by item. P...
198,Can I request a refund if the price drops afte...,If the price of a product drops within 7 days ...


In [24]:
def tokenize_function(examples):
    return tokenizer(examples['question'], examples['answer'], truncation=True)


# Tokenize the dataset
tokenized_dataset = df.map(tokenize_function, batched=True)

tokenized_dataset

Dataset({
    features: ['question', 'answer', 'input_ids', 'attention_mask'],
    num_rows: 200
})

In [25]:
tokenized_dataset.to_pandas()

,question,answer,input_ids,attention_mask
0,How can I create an account?,"To create an account, click on the 'Sign Up' b...","[1128, 508, 306, 1653, 385, 3633, 29973, 1763,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1,What payment methods do you accept?,"We accept major credit cards, debit cards, and...","[1724, 19179, 3519, 437, 366, 3544, 29973, 133...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
2,How can I track my order?,You can track your order by logging into your ...,"[1128, 508, 306, 5702, 590, 1797, 29973, 887, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3,What is your return policy?,Our return policy allows you to return product...,"[1724, 338, 596, 736, 8898, 29973, 8680, 736, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
4,Can I cancel my order?,You can cancel your order if it has not been s...,"[1815, 306, 12611, 590, 1797, 29973, 887, 508,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
...,...,...,...,...
195,Do you offer a satisfaction guarantee?,"Yes, we offer a satisfaction guarantee on our ...","[1938, 366, 5957, 263, 26470, 18818, 29973, 38...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
196,How can I apply for a job at your company?,"To apply for a job at our company, visit our C...","[1128, 508, 306, 3394, 363, 263, 4982, 472, 59...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
197,What is the warranty on your products?,The warranty on our products varies by item. P...,"[1724, 338, 278, 1370, 21867, 29891, 373, 596,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
198,Can I request a refund if the price drops afte...,If the price of a product drops within 7 days ...,"[1815, 306, 2009, 263, 2143, 870, 565, 278, 86...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."


In [26]:
train_test_split = tokenized_dataset.train_test_split(test_size=0.2)
train_dataset = train_test_split['train']
eval_dataset = train_test_split['test']

In [27]:
OUTPUT_DIR = "experiments"
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_seq_length=512,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_8bit",
    eval_strategy="steps",
    eval_steps=0.2,
    save_steps=0.2,
    logging_steps=10,
    learning_rate=1e-4,
    fp16=True,  # or bf16=True,
    save_strategy="steps",
    warmup_ratio=0.1,
    save_total_limit=2,
    lr_scheduler_type="constant",
    report_to="tensorboard",
    save_safetensors=True,
    dataset_kwargs={
        "add_special_tokens": False,  # We template with special tokens
        "append_concat_token": False,  # No need to add additional separator token
    }
)

In [28]:
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer
)

<ipython-input-28-f2e8512b4211>:1: FutureWarning: `tokenizer` is deprecated and removed starting from version 0.16.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  trainer = SFTTrainer(


Converting train dataset to ChatML:   0%|          | 0/160 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/160 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/160 [00:00<?, ? examples/s]

Converting eval dataset to ChatML:   0%|          | 0/40 [00:00<?, ? examples/s]

Applying chat template to eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
8,No log,1.206604
16,1.229100,0.885287
24,0.827700,0.665272
32,0.498100,0.554832
40,0.466600,0.470299


/usr/local/lib/python3.11/dist-packages/peft/utils/save_and_load.py:260: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/peft/utils/save_and_load.py:260: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_r

TrainOutput(global_step=40, training_loss=0.755375599861145, metrics={'train_runtime': 195.8382, 'train_samples_per_second': 1.634, 'train_steps_per_second': 0.204, 'total_flos': 343535150653440.0, 'train_loss': 0.755375599861145})

In [29]:
trainer.evaluate()

{'eval_loss': 1.385213851928711,
 'eval_model_preparation_time': 0.0033,
 'eval_runtime': 6.6915,
 'eval_samples_per_second': 5.978,
 'eval_steps_per_second': 2.989}

In [40]:
def create_prompt(question):
    # Define a template with clear instructions for the model
    prompt = (
        "You are a helpful customer support assistant. "
        "Answer the question clearly and concisely from the provided dataset."
        "Do not repeat the question in your answer.\n\n"
        "Question: " + question + "\n"
        "Answer:"
    )
    return prompt

def generate_response(question):

    prompts = create_prompt(question)
    inputs = tokenizer.encode(question + prompts, return_tensors="pt", padding=True, truncation=True)
    inputs=inputs.to('cuda')

    # Generate the outputs for the question
    outputs = model.generate(
        inputs,
        max_length=100,
        num_beams=4,
        early_stopping=False
    )

    # Decode the outputs, ensuring to skip special tokens
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result

# Example question
question = "What payment methods do you accept?	"
answer = generate_response(question)
print("Question:"+question)
print("Answer:"+answer)

Question:What payment methods do you accept?	
Answer:What payment methods do you accept?	You are a helpful customer support assistant. Answer the question clearly and concisely from the provided dataset.Do not repeat the question in your answer.

Question: What payment methods do you accept?	
Answer: We accept Visa, MasterCard, American Express, and Discover.

Question: What payment methods do you accept?	
Answer: We accept Visa, MasterCard, American Express, and Discover.




In [ ]:

model.save_pretrained("new_model-7B-mini-4k-instruct")  # Save the model
tokenizer.save_pretrained("new_model-7B-mini-4k-instruct")

('new_model-7B-mini-4k-instruct/tokenizer_config.json',
 'new_model-7B-mini-4k-instruct/special_tokens_map.json',
 'new_model-7B-mini-4k-instruct/tokenizer.model',
 'new_model-7B-mini-4k-instruct/added_tokens.json',
 'new_model-7B-mini-4k-instruct/tokenizer.json')